# Multi-objective Bayesian Optimization Tutorial

This notebook walks through the main components of a MOBO pipeline:

1. **Test functions** – multi-objective black-box problems.
2. **GP surrogate** – one independent GP per objective fitted with GPyTorch.
3. **Acquisition functions** – ParEGO, EHVI, MESMO, implemented from scratch.
4. **MOBO loop** – tie everything together and track hypervolume.
5. **Visualisation** – Pareto fronts and hypervolume traces.

All source code lives in `../src/`. Make sure your kernel uses the `uv` environment:

```bash
cd tutorial
uv run jupyter lab
```

In [ ]:
import sys, pathlib

# Add src/ to the path so imports work without installing the package.
src = pathlib.Path("../src").resolve()
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

import torch
import matplotlib.pyplot as plt

print("torch", torch.__version__)
print("src path:", src)

## 1. Test functions

Each test function is a callable `f(x)` that maps `(n, d)` inputs to `(n, M)` objectives.
All objectives use the **maximisation** convention.

Available functions: **Sinusoidal** (d=1, M=2), **BraninCurrin** (d=2, M=2), **ZDT1** (d=6, M=2), **DTLZ2** (d=6, M=2).

In [ ]:
from test_functions import get_test_function, REGISTRY as TEST_FN_REGISTRY

print("Available test functions:", list(TEST_FN_REGISTRY.keys()))

# Instantiate and inspect BraninCurrin
fn = get_test_function("DTLZ2")
print(fn)
print("Search space bounds:\n", fn.search_space.bounds)
print("Reference point:", fn.ref_point)

# Quick sanity-check: evaluate 5 random points
x_test = torch.rand(5, fn.d)
y_test = fn(x_test)
print("\nRandom evaluations (shape):", y_test.shape)
print(y_test)

## 2. GP surrogate

We fit one independent GP per objective using GPyTorch with a Matérn-5/2 kernel.
Hyperparameters are optimised via marginal-likelihood maximisation (L-BFGS).

In [ ]:
from surrogate import fit_gp, predict

# Generate a small training set
fn = get_test_function("BraninCurrin")
torch.manual_seed(0)
X_train = torch.rand(3, fn.d)
Y_train = fn(X_train)

# Standardise for GP stability
Y_mean, Y_std = Y_train.mean(0), Y_train.std(0).clamp(min=1e-6)
Y_norm = (Y_train - Y_mean) / Y_std

# Fit one GP per objective
models = fit_gp(X_train, Y_norm, n_iter=100)
print(f"Fitted {len(models)} GPs")

# Predict on a grid
x_grid = torch.rand(200, fn.d)
mu, sigma = predict(models, x_grid)
print("Predictive mean shape:", mu.shape)
print("Predictive std  shape:", sigma.shape)

## 3. Acquisition functions

We implement three acquisition functions from scratch:

| Name | Key idea |
|------|----------|
| **ParEGO** | Random Chebyshev scalarization → analytic EI on a single GP |
| **EHVI** | Expected Hypervolume Improvement, analytic (M=2) or MC (M>2) |
| **MESMO** | Information gain about the Pareto front via Pareto set samples |

In [ ]:
from acq import get_acquisition, REGISTRY as ACQ_REGISTRY
from hypervolume import _nondominated, hypervolume

print("Available acquisition functions:", list(ACQ_REGISTRY.keys()))

# Example: evaluate each acquisition on random candidates
pareto_y = _nondominated(Y_norm)[0]
ref_norm = (fn.ref_point - Y_mean) / Y_std
bounds_norm = torch.stack([
    (torch.zeros(fn.d) - X_train.mean(0)) / X_train.std(0).clamp(min=1e-6),
    (torch.ones(fn.d)  - X_train.mean(0)) / X_train.std(0).clamp(min=1e-6),
])
x_cands = torch.rand(8, fn.d)

for acq_name in ACQ_REGISTRY:
    if acq_name == "EHVI":
        acq = get_acquisition(acq_name, models=models, pareto_y=pareto_y, ref_point=ref_norm)
    elif acq_name == "MESMO":
        acq = get_acquisition(acq_name, models=models, bounds=bounds_norm, n_grid=64)
    elif acq_name == "ParEGO":  # ParEGO
        acq = get_acquisition(acq_name, models=models, train_y=Y_norm)
    vals = acq(x_cands)
    print(f"{acq_name:8s}  values: {vals.detach().numpy().round(4)}")

## 4. Running the MOBO loop

Select a **test function** and an **acquisition function**, then run for N iterations.
Hypervolume is tracked after every evaluation.

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
TEST_FUNCTION = "BraninCurrin"   # "BraninCurrin" | "ZDT1" | "DTLZ2"
ACQUISITION   = "EHVI"           # "EHVI" | "ParEGO" | "MESMO"
N_INIT        = 3
N_ITER        = 15
SEED          = 42
# ─────────────────────────────────────────────────────────────────────────────


from mobo import run_mobo

fn = get_test_function(TEST_FUNCTION)

result = run_mobo(
    test_fn=fn,
    acq_name=ACQUISITION,
    n_init=N_INIT,
    n_iter=N_ITER,
    n_restarts=10,
    n_raw=256,
    gp_n_iter=80,
    seed=SEED,
    verbose=True,
)
print(f"\nFinal hypervolume: {result.final_hv:.4f}")

## 5. Visualisation

### 5a. Hypervolume trace

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(result.hv_trace, marker="o", markersize=4, linewidth=1.8, color="#0f766e")
ax.axvline(N_INIT, color="grey", linestyle=":", linewidth=1.2, label=f"end of init (n={N_INIT})")
ax.set_xlabel("Evaluation index")
ax.set_ylabel("Hypervolume")
ax.set_title(f"{TEST_FUNCTION} + {ACQUISITION}")
ax.legend()
fig.tight_layout()
plt.show()

### 5b. Pareto front in objective space

In [ ]:
from hypervolume import _nondominated

Y_all = result.Y.detach()
Y_pareto = _nondominated(Y_all)[0].detach()

# Sort Pareto front for step plot
order = Y_pareto[:, 0].argsort()
Y_pareto_sorted = Y_pareto[order]

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(Y_all[:, 0], Y_all[:, 1], c="steelblue", alpha=0.4, s=25, label="all evaluations")
ax.scatter(Y_pareto[:, 0], Y_pareto[:, 1], c="#0f766e", s=60, zorder=5, label="Pareto front")
ax.scatter(fn.ref_point[0], fn.ref_point[1], c="#241a74", s=60, zorder=5, label="ref point")

ax.step(
    Y_pareto_sorted[:, 0].numpy(),
    Y_pareto_sorted[:, 1].numpy(),
    where="post", color="#0f766e", linewidth=1.4, alpha=0.6,
)
ax.set_xlabel("Objective 1 (↑)")
ax.set_ylabel("Objective 2 (↑)")
ax.set_title(f"Pareto front – {TEST_FUNCTION} + {ACQUISITION}")
ax.legend()
fig.tight_layout()
plt.show()

## 6. Benchmark (all acquisitions × all test functions)

> **Note:** This section is computationally intensive.  
> For a quick test set `N_BENCH_ITER = 5` and `N_SEEDS = 1`.
> For the full comparison use the standalone script:
> ```bash
> cd tutorial
> uv run python src/benchmark.py --n_iter 20 --n_seeds 5
> ```

In [ ]:
import time
from collections import defaultdict

# ── Benchmark configuration ───────────────────────────────────────────────────
N_BENCH_INIT  = 5
N_BENCH_ITER  = 20    # increase for a proper comparison
N_SEEDS       = 5    # increase for smoother curves
BENCH_FNS     = ["BraninCurrin"]          # add "ZDT1", "DTLZ2" as desired
BENCH_ACQS    = list(ACQ_REGISTRY.keys())
# ─────────────────────────────────────────────────────────────────────────────

results_bench = {}   # (fn_name, acq_name, seed) -> MOBOResult

for fn_name in BENCH_FNS:
    for acq_name in BENCH_ACQS:
        for seed in range(N_SEEDS):
            key = (fn_name, acq_name, seed)
            print(f"  {fn_name} + {acq_name} seed={seed} ...", end=" ", flush=True)
            t0 = time.perf_counter()
            results_bench[key] = run_mobo(
                test_fn=get_test_function(fn_name),
                acq_name=acq_name,
                n_init=N_BENCH_INIT,
                n_iter=N_BENCH_ITER,
                n_restarts=8,
                n_raw=128,
                gp_n_iter=50,
                seed=seed,
                verbose=False,
            )
            print(f"done ({time.perf_counter()-t0:.1f}s)")

In [ ]:
colors = {"EHVI": "#0f766e", "ParEGO": "#1d4ed8", "MESMO": "#b45309"}
n_fns = len(BENCH_FNS)
fig, axes = plt.subplots(1, n_fns, figsize=(5.5 * n_fns, 4), squeeze=False)

for col, fn_name in enumerate(BENCH_FNS):
    ax = axes[0][col]
    for acq_name in BENCH_ACQS:
        traces = [results_bench[(fn_name, acq_name, s)].hv_trace for s in range(N_SEEDS)]
        traces_t = torch.tensor(traces, dtype=torch.float32)
        mean = traces_t.mean(0).numpy()
        std  = traces_t.std(0).numpy()
        xs = range(len(mean))
        c = colors.get(acq_name, "grey")
        ax.plot(xs, mean, label=acq_name, color=c, linewidth=2)
        ax.fill_between(xs, mean - std, mean + std, alpha=0.15, color=c)
    ax.axvline(N_BENCH_INIT, color="grey", linestyle=":", linewidth=1.1, label="end of init")
    ax.set_title(fn_name)
    ax.set_xlabel("Evaluation index")
    ax.set_ylabel("Hypervolume")
    ax.legend(fontsize=9)

fig.suptitle("Benchmark – hypervolume vs. evaluations", y=1.02)
fig.tight_layout()
plt.show()

# Wall-clock summary
print("\nMean iteration time (seconds):")
print(f"{'Acquisition':10s}", end="")
for fn_name in BENCH_FNS:
    print(f"  {fn_name:>15s}", end="")
print()
for acq_name in BENCH_ACQS:
    print(f"{acq_name:10s}", end="")
    for fn_name in BENCH_FNS:
        all_times = []
        for seed in range(N_SEEDS):
            all_times.extend(results_bench[(fn_name, acq_name, seed)].iter_times)
        mean_t = sum(all_times) / len(all_times)
        print(f"  {mean_t:>15.2f}", end="")
    print()